In [ ]:
%pip install --quiet pandas pyarrow geopandas folium branca matplotlib shapely

# Visualização — Pipeline Brasil (mapas folium otimizados)

Três mapas complementares lendo o output do `Notebooks_Final/4_pipeline_etl_brasil.ipynb` (`saida_etl_final_brasil/renda_setor_cep_brasil_final.parquet`):

| Mapa | O que mostra | Granularidade | Escalabilidade |
|---|---|---|---|
| **A) Brasil agregado** | Choropleth de renda média + cobertura CEP **por município** | ~5.570 municípios (Brasil full) ou só os do recorte (smoke) | Geometria simplificada (tolerance 0,005°) — HTML ~5-15 MB |
| **B) Drill UF** | Setores de uma UF inteira (escolhida em `UF_RECORTE`) | 7k-25k setores | HTML 10-50 MB |
| **C) Drill município** | Setores de 1 município (escolhido em `MUN_RECORTE`) — mesmo padrão visual do `3_visualizacao_mapa.ipynb` | 100-30k setores | HTML 0,5-100 MB |

**Outputs (salvos automaticamente em `saida_etl_final_brasil/mapas/`):**
- `mapa_brasil_agregado.html`
- `mapa_uf_<SIGLA>.html`
- `mapa_municipio_<COD_MUN>.html`

**Funciona tanto com smoke quanto com full Brasil** — basta re-rodar este notebook quando o parquet for atualizado.

## Setup — imports, paths e configuração de recorte

In [ ]:
from pathlib import Path
import json

import pandas as pd
import geopandas as gpd
import folium
import branca.colormap as cm
from shapely.geometry import mapping
from IPython.display import display

pd.set_option('display.max_columns', 120)
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')

In [ ]:
def find_project_root(start):
    start = Path(start).resolve()
    for c in [start, *start.parents]:
        if (c / 'saida_etl_final_brasil').exists() and (c / 'BR_setores_CD2022').exists():
            return c
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
PARQUET      = PROJECT_ROOT / 'saida_etl_final_brasil' / 'renda_setor_cep_brasil_final.parquet'
SHAPEFILE    = PROJECT_ROOT / 'BR_setores_CD2022' / 'BR_setores_CD2022.shp'
MAPAS_DIR    = PROJECT_ROOT / 'saida_etl_final_brasil' / 'mapas'
MAPAS_DIR.mkdir(parents=True, exist_ok=True)

for p in [PARQUET, SHAPEFILE]:
    print(f'{p.name}: {"OK" if p.exists() else "NAO ENCONTRADO"}')
print(f'MAPAS_DIR: {MAPAS_DIR}')

In [ ]:
# ===========================================================================
# CONFIGURACAO DE RECORTE (mude conforme o parquet disponivel)
# ===========================================================================

# Mapa A: simplificacao da geometria (em graus; 0.005 = ~500m no equador).
# Maior = HTML menor + mapa mais 'feio' no zoom. 0.005 a 0.01 sao bons defaults.
SIMPLIFY_TOLERANCE = 0.005

# Mapa B: UF pra mostrar todos os setores (sigla).
UF_RECORTE = 'RO'

# Mapa C: municipio pra mostrar setores detalhados (cod_mun IBGE de 7 digitos).
# Default: Porto Velho (RO) = '1100205'. Outras opcoes do smoke:
#   Rio Branco (AC) = '1200401'  |  Boa Vista (RR) = '1400100'
MUN_RECORTE = '1100205'

# Cor da renda
COLORMAP_RENDA = cm.linear.YlOrRd_09

print({
    'SIMPLIFY_TOLERANCE': SIMPLIFY_TOLERANCE,
    'UF_RECORTE': UF_RECORTE,
    'MUN_RECORTE': MUN_RECORTE,
})

## Carregar parquet + funções auxiliares

In [ ]:
df = pd.read_parquet(PARQUET)
print(f'Linhas no parquet: {len(df):,}')
print(f'UFs no parquet:    {sorted(df["sigla_uf"].dropna().unique().tolist())}')
print(f'Total municipios:  {df["cod_mun"].nunique():,}')


def fmt_brl(v):
    if v is None or pd.isna(v):
        return '-'
    s = f'{float(v):,.2f}'
    return 'R$ ' + s.replace(',', 'X').replace('.', ',').replace('X', '.')


def fmt_int(v):
    if v is None or pd.isna(v):
        return '-'
    return f'{int(v):,}'.replace(',', '.')


def fmt_pct(v):
    if v is None or pd.isna(v):
        return '-'
    return f'{float(v):.2f}%'


def short_ceps(s):
    if s is None or (isinstance(s, float) and pd.isna(s)) or not s:
        return ''
    ceps = str(s).split('|')
    if len(ceps) <= 5:
        return ', '.join(ceps)
    return ', '.join(ceps[:5]) + f'  ... (+{len(ceps)-5})'


def sanitize_for_geojson(gdf):
    """NaN -> None pra colunas numericas (folium serializa como null no JSON),
    "" pra colunas textuais. Evita o bug 'str <= float' no _repr_html_."""
    num_cols = list(gdf.select_dtypes(include=['number']).columns)
    for col in num_cols:
        gdf[col] = gdf[col].astype(object).where(gdf[col].notna(), None)
    str_cols = [c for c in gdf.columns
                if c not in num_cols
                and c != 'geometry'
                and pd.api.types.is_object_dtype(gdf[c])]
    for col in str_cols:
        gdf[col] = gdf[col].fillna('')
    return gdf

## Mapa A — Brasil agregado por município

Agrega o parquet por `cod_mun` (renda média simples, cobertura CEP, contagem de setores) e desenha um choropleth folium com **geometria simplificada** (`shapely.simplify`) — viável até pro Brasil completo (~5.570 municípios).

In [ ]:
# Agrega parquet por municipio.
mun_stats = (
    df.groupby(['cod_mun', 'nm_mun', 'sigla_uf', 'nm_uf'], dropna=False)
      .agg(setores=('cd_setor', 'count'),
           com_cep=('tem_cep', 'sum'),
           com_renda=('renda_v06004_estimada', lambda s: s.notna().sum()),
           renda_media=('renda_v06004_estimada', 'mean'),
           renda_mediana=('renda_v06006_estimada', 'median'))
      .reset_index()
)
mun_stats['pct_cep']   = (mun_stats['com_cep']   / mun_stats['setores'] * 100).round(2)
mun_stats['pct_renda'] = (mun_stats['com_renda'] / mun_stats['setores'] * 100).round(2)
print(f'Municipios agregados: {len(mun_stats):,}')
display(mun_stats.head(5))

# Carrega geometria SO dos municipios presentes no parquet.
codigos_mun = set(mun_stats['cod_mun'].astype(str))
print(f'\nCarregando geometria do shapefile (filtrando {len(codigos_mun):,} municipios)...')
shp = gpd.read_file(SHAPEFILE, columns=['CD_SETOR', 'CD_MUN', 'geometry'])
shp['cd_setor'] = shp['CD_SETOR'].astype(str).str.strip()
shp['cod_mun']  = shp['CD_MUN'].astype(str).str.zfill(7)
shp = shp.loc[shp['cod_mun'].isin(codigos_mun)]
print(f'Setores carregados (pre-dissolve): {len(shp):,}')

# Simplify + dissolve por municipio.
print(f'Simplificando geometrias (tolerance={SIMPLIFY_TOLERANCE})...')
shp['geometry'] = shp.geometry.simplify(SIMPLIFY_TOLERANCE, preserve_topology=True)
print('Dissolvendo por municipio...')
mun_geo = shp.dissolve(by='cod_mun', as_index=False)[['cod_mun', 'geometry']]
print(f'Municipios com geometria: {len(mun_geo):,}')

# Merge stats + geo.
mun_map = mun_geo.merge(mun_stats, on='cod_mun', how='inner')
mun_map = sanitize_for_geojson(mun_map)
if mun_map.crs is not None and mun_map.crs.to_epsg() != 4326:
    mun_map = mun_map.to_crs(epsg=4326)
print(f'\nPronto pro mapa: {len(mun_map):,} municipios')

In [ ]:
# Tooltip pre-formatado.
mun_map['Municipio_lbl']    = mun_map['nm_mun'].astype(str)
mun_map['UF_lbl']           = mun_map['sigla_uf'].astype(str)
mun_map['Setores_lbl']      = mun_map['setores'].apply(fmt_int)
mun_map['ComCEP_lbl']       = mun_map['com_cep'].apply(fmt_int)
mun_map['PctCEP_lbl']       = mun_map['pct_cep'].apply(fmt_pct)
mun_map['ComRenda_lbl']     = mun_map['com_renda'].apply(fmt_int)
mun_map['PctRenda_lbl']     = mun_map['pct_renda'].apply(fmt_pct)
mun_map['RendaMedia_lbl']   = mun_map['renda_media'].apply(fmt_brl)
mun_map['RendaMediana_lbl'] = mun_map['renda_mediana'].apply(fmt_brl)

# Colormap pela renda media.
renda_vals = pd.to_numeric(pd.Series([v for v in mun_map['renda_media'] if v is not None]),
                            errors='coerce').dropna()
if len(renda_vals):
    cmap = COLORMAP_RENDA.scale(float(renda_vals.quantile(0.05)),
                                 float(renda_vals.quantile(0.95)))
    cmap.caption = 'Renda media mensal por municipio (R$)'
else:
    cmap = None


def style_a(feature):
    p = feature['properties']
    r = p.get('renda_media')
    return {
        'fillColor': '#cccccc' if r is None else (cmap(r) if cmap else '#999999'),
        'color':     '#666',
        'weight':    0.4,
        'fillOpacity': 0.75,
    }


# Centro do mapa.
center = mun_map.geometry.union_all().centroid
m_a = folium.Map(location=[center.y, center.x], zoom_start=5, tiles='cartodbpositron', control_scale=True)

folium.GeoJson(
    mun_map,
    name='Municipios',
    style_function=style_a,
    tooltip=folium.features.GeoJsonTooltip(
        fields=['Municipio_lbl','UF_lbl','Setores_lbl','ComCEP_lbl','PctCEP_lbl',
                'ComRenda_lbl','PctRenda_lbl','RendaMedia_lbl','RendaMediana_lbl'],
        aliases=['Municipio:','UF:','Setores:','com CEP:','% CEP:',
                 'com renda:','% renda:','Renda media:','Renda mediana:'],
        sticky=True, max_width=380,
    ),
).add_to(m_a)

if cmap is not None:
    cmap.add_to(m_a)

folium.LayerControl(collapsed=False).add_to(m_a)

# Salva HTML standalone.
mapa_a_path = MAPAS_DIR / 'mapa_brasil_agregado.html'
m_a.save(str(mapa_a_path))
print(f'Mapa A salvo: {mapa_a_path}  ({mapa_a_path.stat().st_size/1e6:.1f} MB)')
m_a

## Mapa B — Drill UF (todos os setores de uma UF)

Configurado por `UF_RECORTE` no início. Mostra todos os setores da UF, colorindo por renda. Geometria sem simplify (precisão preservada).

In [ ]:
recorte_uf = df.loc[df['sigla_uf'] == UF_RECORTE].copy()
if len(recorte_uf) == 0:
    raise ValueError(f'UF_RECORTE={UF_RECORTE!r} nao existe no parquet. UFs disponiveis: {sorted(df["sigla_uf"].dropna().unique().tolist())}')
print(f'Setores na UF {UF_RECORTE}: {len(recorte_uf):,}')
print(f'  com renda: {int(recorte_uf["renda_v06004_estimada"].notna().sum()):,}')
print(f'  com CEP:   {int((recorte_uf["tem_cep"] == 1).sum()):,}')

# Codigo UF a partir do parquet (ja preenchido).
cod_uf_recorte = recorte_uf['cod_uf'].iloc[0]
print(f'cod_uf={cod_uf_recorte}')

print(f'\nCarregando geometria dos setores da UF {UF_RECORTE}...')
geo_uf = gpd.read_file(SHAPEFILE, where=f"CD_UF = '{cod_uf_recorte}'")
geo_uf['cd_setor'] = geo_uf['CD_SETOR'].astype(str).str.strip()
print(f'Setores no shapefile: {len(geo_uf):,}')

g_uf = geo_uf[['cd_setor', 'geometry']].merge(recorte_uf, on='cd_setor', how='left')
if g_uf.crs is not None and g_uf.crs.to_epsg() != 4326:
    g_uf = g_uf.to_crs(epsg=4326)
print(f'Linhas merge: {len(g_uf):,}')

In [ ]:
# Tooltip basico (UF tem muito setor; tooltip rico vai pro mapa C).
g_uf['Setor_lbl']      = g_uf['cd_setor'].astype(str)
g_uf['Municipio_lbl']  = g_uf['nm_mun'].fillna('').astype(str)
g_uf['Renda_lbl']      = g_uf['renda_v06004_estimada'].apply(fmt_brl)
g_uf['Origem_lbl']     = g_uf['origem_renda'].fillna('').astype(str)
g_uf['OrigemCep_lbl']  = g_uf['origem_cep'].fillna('').astype(str)

g_uf = sanitize_for_geojson(g_uf)

renda_vals_uf = pd.to_numeric(pd.Series([v for v in g_uf['renda_v06004_estimada'] if v is not None]),
                                errors='coerce').dropna()
if len(renda_vals_uf):
    cmap_uf = COLORMAP_RENDA.scale(float(renda_vals_uf.quantile(0.05)),
                                    float(renda_vals_uf.quantile(0.95)))
    cmap_uf.caption = f'Renda media setor (R$) - {UF_RECORTE}'
else:
    cmap_uf = None


def style_b(feature):
    p = feature['properties']
    r = p.get('renda_v06004_estimada')
    origem = p.get('origem_cep') or ''
    fill = '#cccccc' if r is None else (cmap_uf(r) if cmap_uf else '#999999')
    border_color = {'geofencing': 'black', 'cnefe_original': 'blue',
                    'sem_endereco_cnefe': 'red'}.get(origem, 'gray')
    border_weight = {'geofencing': 0.3, 'cnefe_original': 0.8,
                     'sem_endereco_cnefe': 1.8}.get(origem, 0.5)
    return {
        'fillColor': fill,
        'color':     border_color,
        'weight':    border_weight,
        'fillOpacity': 0.75,
    }


center_uf = g_uf.geometry.union_all().centroid
m_b = folium.Map(location=[center_uf.y, center_uf.x], zoom_start=7,
                 tiles='cartodbpositron', control_scale=True)

folium.GeoJson(
    g_uf,
    name=f'Setores {UF_RECORTE}',
    style_function=style_b,
    tooltip=folium.features.GeoJsonTooltip(
        fields=['Setor_lbl','Municipio_lbl','Renda_lbl','Origem_lbl','OrigemCep_lbl'],
        aliases=['Setor:','Municipio:','Renda media:','Origem renda:','Origem CEP:'],
        sticky=True, max_width=360,
    ),
).add_to(m_b)

if cmap_uf is not None:
    cmap_uf.add_to(m_b)

folium.LayerControl(collapsed=False).add_to(m_b)

mapa_b_path = MAPAS_DIR / f'mapa_uf_{UF_RECORTE}.html'
m_b.save(str(mapa_b_path))
print(f'Mapa B salvo: {mapa_b_path}  ({mapa_b_path.stat().st_size/1e6:.1f} MB)')
m_b

## Mapa C — Drill município (estilo do `3_visualizacao_mapa.ipynb`)

Configurado por `MUN_RECORTE`. Tooltip completo com todos os campos do setor — Bairro, Distrito, CD_TIPO, Renda média e mediana, Origem renda/CEP, CEPs, Endereços, etc.

In [ ]:
recorte_mun = df.loc[df['cod_mun'] == MUN_RECORTE].copy()
if len(recorte_mun) == 0:
    munis_disponiveis = (df[['cod_mun','nm_mun','sigla_uf']].drop_duplicates()
                          .sort_values(['sigla_uf','nm_mun']).head(20))
    print(f'MUN_RECORTE={MUN_RECORTE!r} nao existe no parquet.')
    print('Primeiros 20 disponiveis:')
    print(munis_disponiveis.to_string(index=False))
    raise ValueError(f'cod_mun {MUN_RECORTE} ausente')

print(f'Municipio: {recorte_mun["nm_mun"].iloc[0]} / {recorte_mun["sigla_uf"].iloc[0]}')
print(f'Setores: {len(recorte_mun):,}')
print(f'  com renda: {int(recorte_mun["renda_v06004_estimada"].notna().sum()):,}')
print(f'  com CEP:   {int((recorte_mun["tem_cep"] == 1).sum()):,}')

print(f'\nCarregando geometria dos setores do municipio {MUN_RECORTE}...')
geo_mun = gpd.read_file(SHAPEFILE, where=f"CD_MUN = '{MUN_RECORTE}'")
geo_mun['cd_setor'] = geo_mun['CD_SETOR'].astype(str).str.strip()
print(f'Geometrias: {len(geo_mun):,}')

g_mun = geo_mun[['cd_setor', 'geometry']].merge(recorte_mun, on='cd_setor', how='left')
if g_mun.crs is not None and g_mun.crs.to_epsg() != 4326:
    g_mun = g_mun.to_crs(epsg=4326)
print(f'Linhas merge: {len(g_mun):,}')

In [ ]:
# Tooltip rico (mesmo padrao do nb 3).
g_mun['Setor']          = g_mun['cd_setor'].astype(str)
g_mun['Municipio']      = g_mun['nm_mun'].fillna('').astype(str)
g_mun['Distrito']       = g_mun['NM_DIST'].fillna('').astype(str)
g_mun['Bairro']         = g_mun['NM_BAIRRO'].fillna('').astype(str)
g_mun['CD_TIPO_lbl']    = g_mun['CD_TIPO'].fillna('').astype(str)
g_mun['Situacao']       = g_mun['SITUACAO'].fillna('').astype(str)
g_mun['Renda media']    = g_mun['renda_v06004_estimada'].apply(fmt_brl)
g_mun['Renda mediana']  = g_mun['renda_v06006_estimada'].apply(fmt_brl)
g_mun['Origem renda']   = g_mun['origem_renda'].fillna('').astype(str)
g_mun['Origem CEP']     = g_mun['origem_cep'].fillna('').astype(str)
g_mun['Qtd CEPs']       = g_mun['qtd_ceps'].apply(fmt_int)
g_mun['Faixa CEP']      = g_mun['faixa_cep'].fillna('').astype(str)
g_mun['Enderecos']      = g_mun['total_enderecos'].apply(fmt_int)
g_mun['CEPs']           = g_mun['lista_ceps'].apply(short_ceps)

g_mun = sanitize_for_geojson(g_mun)

renda_vals_mun = pd.to_numeric(pd.Series([v for v in g_mun['renda_v06004_estimada'] if v is not None]),
                                errors='coerce').dropna()
if len(renda_vals_mun):
    cmap_mun = COLORMAP_RENDA.scale(float(renda_vals_mun.min()), float(renda_vals_mun.max()))
    cmap_mun.caption = f'Renda media setor (R$) - {recorte_mun["nm_mun"].iloc[0]}'
else:
    cmap_mun = None

ORIGEM_STYLE = {
    'geofencing':         {'color': 'black', 'weight': 0.4},
    'cnefe_original':     {'color': 'blue',  'weight': 1.2},
    'sem_endereco_cnefe': {'color': 'red',   'weight': 2.5},
}
DEFAULT_BORDER = {'color': 'gray', 'weight': 1.0}


def style_c(feature):
    p = feature['properties']
    r = p.get('renda_v06004_estimada')
    origem = p.get('origem_cep') or ''
    fill = '#cccccc' if r is None else (cmap_mun(r) if cmap_mun else '#999999')
    s = ORIGEM_STYLE.get(origem, DEFAULT_BORDER)
    return {
        'fillColor': fill,
        'color':     s['color'],
        'weight':    s['weight'],
        'fillOpacity': 0.75,
    }


center_mun = g_mun.geometry.union_all().centroid
m_c = folium.Map(location=[center_mun.y, center_mun.x], zoom_start=12,
                 tiles='cartodbpositron', control_scale=True)

tooltip_pairs = [
    ('Setor', 'Setor:'), ('Municipio', 'Municipio:'), ('Distrito', 'Distrito:'),
    ('Bairro', 'Bairro:'), ('Situacao', 'Situacao:'), ('CD_TIPO_lbl', 'CD_TIPO:'),
    ('Renda media', 'Renda media:'), ('Renda mediana', 'Renda mediana:'),
    ('Origem renda', 'Origem renda:'), ('Origem CEP', 'Origem CEP:'),
    ('Qtd CEPs', 'Qtd CEPs:'), ('Faixa CEP', 'Faixa CEP:'),
    ('Enderecos', 'Enderecos:'), ('CEPs', 'CEPs:'),
]
fields  = [f for f, _ in tooltip_pairs if f in g_mun.columns]
aliases = [a for f, a in tooltip_pairs if f in g_mun.columns]

folium.GeoJson(
    g_mun, name='Setores',
    style_function=style_c,
    tooltip=folium.features.GeoJsonTooltip(fields=fields, aliases=aliases,
                                            sticky=True, max_width=420),
).add_to(m_c)

if cmap_mun is not None:
    cmap_mun.add_to(m_c)

# Legenda de bordas.
legend_html = '''
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999;
            background: white; padding: 10px; border: 2px solid #888;
            font-family: sans-serif; font-size: 12px; line-height: 18px;">
  <b>Origem do CEP</b><br>
  <span style="display:inline-block; width:18px; height:3px; background:black; vertical-align:middle;"></span> geofencing<br>
  <span style="display:inline-block; width:18px; height:3px; background:blue;  vertical-align:middle;"></span> cnefe_original<br>
  <span style="display:inline-block; width:18px; height:3px; background:red;   vertical-align:middle;"></span> sem_endereco_cnefe<br>
  <hr style="margin: 4px 0;">
  <b>Preenchimento</b><br>
  cinza = sem renda (legitimo ou nao imputavel)<br>
  gradiente = renda media (V06004)
</div>
'''
m_c.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl(collapsed=False).add_to(m_c)

mapa_c_path = MAPAS_DIR / f'mapa_municipio_{MUN_RECORTE}.html'
m_c.save(str(mapa_c_path))
print(f'Mapa C salvo: {mapa_c_path}  ({mapa_c_path.stat().st_size/1e6:.1f} MB)')
m_c

## Mapa D — SP-capital (bônus, lê parquet do `1_pipeline_etl_sp.ipynb`)

Bônus pra visualizar São Paulo capital **antes do Brasil full rodar**. Lê de outro parquet — `saida_etl_final_sp/renda_setor_cep_sp_final.parquet` — usando o mesmo estilo do Mapa C.

Útil pra:
- Comparar visualmente Porto Velho (smoke, ~780 setores) com SP-capital (~27k setores)
- Validar que o estilo dashboard escala bem
- Servir de "preview" do que vai funcionar quando o full Brasil for rodado e você puder usar `MUN_RECORTE='3550308'` no Mapa C

**Reaproveita** as funções (`fmt_brl`, `fmt_int`, `short_ceps`, `sanitize_for_geojson`), `COLORMAP_RENDA`, `ORIGEM_STYLE`, `DEFAULT_BORDER` e `legend_html` já definidos acima — rode as células anteriores primeiro.


In [ ]:
# Config separada do Mapa D (le parquet diferente do nb 1).
PARQUET_SP = PROJECT_ROOT / 'saida_etl_final_sp' / 'renda_setor_cep_sp_final.parquet'

# Municipio default: SP-capital (cod_mun IBGE = 3550308). Outras opcoes:
#   '3509502' Campinas  |  '3518800' Guarulhos  |  '3548708' Sao Bernardo do Campo
#   '3547809' Santo Andre  |  '3543402' Ribeirao Preto  |  '3525904' Jundiai
MUN_SP = '3550308'

print(f'PARQUET_SP: {PARQUET_SP.name} -> {"OK" if PARQUET_SP.exists() else "NAO ENCONTRADO"}')
print(f'MUN_SP: {MUN_SP}')


In [ ]:
df_sp = pd.read_parquet(PARQUET_SP)
print(f'Parquet SP carregado: {len(df_sp):,} linhas')
print(f'  UFs: {sorted(df_sp["sigla_uf"].dropna().unique().tolist())}')
print(f'  municipios: {df_sp["cod_mun"].nunique():,}')

recorte_sp = df_sp.loc[df_sp['cod_mun'] == MUN_SP].copy()
if len(recorte_sp) == 0:
    munis = (df_sp[['cod_mun','nm_mun']].drop_duplicates()
              .sort_values('nm_mun').head(20))
    print(f'\nMUN_SP={MUN_SP!r} ausente. Primeiros 20:')
    print(munis.to_string(index=False))
    raise ValueError(f'cod_mun {MUN_SP} nao encontrado')

print(f'\nMunicipio: {recorte_sp["nm_mun"].iloc[0]} / {recorte_sp["sigla_uf"].iloc[0]}')
print(f'Setores: {len(recorte_sp):,}')
print(f'  com renda: {int(recorte_sp["renda_v06004_estimada"].notna().sum()):,}')
print(f'  com CEP:   {int((recorte_sp["tem_cep"] == 1).sum()):,}')

print(f'\nCarregando geometria dos setores do municipio {MUN_SP}...')
geo_sp = gpd.read_file(SHAPEFILE, where=f"CD_MUN = '{MUN_SP}'")
geo_sp['cd_setor'] = geo_sp['CD_SETOR'].astype(str).str.strip()
print(f'Geometrias: {len(geo_sp):,}')

g_sp = geo_sp[['cd_setor', 'geometry']].merge(recorte_sp, on='cd_setor', how='left')
if g_sp.crs is not None and g_sp.crs.to_epsg() != 4326:
    g_sp = g_sp.to_crs(epsg=4326)
print(f'Linhas merge: {len(g_sp):,}')


In [ ]:
# Tooltip rico (mesmo padrao do Mapa C).
g_sp['Setor']          = g_sp['cd_setor'].astype(str)
g_sp['Municipio']      = g_sp['nm_mun'].fillna('').astype(str)
g_sp['Distrito']       = g_sp['NM_DIST'].fillna('').astype(str)
g_sp['Bairro']         = g_sp['NM_BAIRRO'].fillna('').astype(str)
g_sp['CD_TIPO_lbl']    = g_sp['CD_TIPO'].fillna('').astype(str)
g_sp['Situacao']       = g_sp['SITUACAO'].fillna('').astype(str)
g_sp['Renda media']    = g_sp['renda_v06004_estimada'].apply(fmt_brl)
g_sp['Renda mediana']  = g_sp['renda_v06006_estimada'].apply(fmt_brl)
g_sp['Origem renda']   = g_sp['origem_renda'].fillna('').astype(str)
g_sp['Origem CEP']     = g_sp['origem_cep'].fillna('').astype(str)
g_sp['Qtd CEPs']       = g_sp['qtd_ceps'].apply(fmt_int)
g_sp['Faixa CEP']      = g_sp['faixa_cep'].fillna('').astype(str)
g_sp['Enderecos']      = g_sp['total_enderecos'].apply(fmt_int)
g_sp['CEPs']           = g_sp['lista_ceps'].apply(short_ceps)

g_sp = sanitize_for_geojson(g_sp)

# Colormap pelo intervalo 5-95% pra evitar outliers achatarem a escala.
renda_vals_sp = pd.to_numeric(pd.Series([v for v in g_sp['renda_v06004_estimada'] if v is not None]),
                                errors='coerce').dropna()
if len(renda_vals_sp):
    cmap_sp = COLORMAP_RENDA.scale(float(renda_vals_sp.quantile(0.05)),
                                    float(renda_vals_sp.quantile(0.95)))
    cmap_sp.caption = f'Renda media setor (R$) - {recorte_sp["nm_mun"].iloc[0]}'
else:
    cmap_sp = None


def style_d(feature):
    p = feature['properties']
    r = p.get('renda_v06004_estimada')
    origem = p.get('origem_cep') or ''
    fill = '#cccccc' if r is None else (cmap_sp(r) if cmap_sp else '#999999')
    s = ORIGEM_STYLE.get(origem, DEFAULT_BORDER)
    return {
        'fillColor': fill,
        'color':     s['color'],
        'weight':    s['weight'],
        'fillOpacity': 0.75,
    }


center_sp = g_sp.geometry.union_all().centroid
m_d = folium.Map(location=[center_sp.y, center_sp.x], zoom_start=11,
                 tiles='cartodbpositron', control_scale=True)

tooltip_pairs_d = [
    ('Setor', 'Setor:'), ('Municipio', 'Municipio:'), ('Distrito', 'Distrito:'),
    ('Bairro', 'Bairro:'), ('Situacao', 'Situacao:'), ('CD_TIPO_lbl', 'CD_TIPO:'),
    ('Renda media', 'Renda media:'), ('Renda mediana', 'Renda mediana:'),
    ('Origem renda', 'Origem renda:'), ('Origem CEP', 'Origem CEP:'),
    ('Qtd CEPs', 'Qtd CEPs:'), ('Faixa CEP', 'Faixa CEP:'),
    ('Enderecos', 'Enderecos:'), ('CEPs', 'CEPs:'),
]
fields_d  = [f for f, _ in tooltip_pairs_d if f in g_sp.columns]
aliases_d = [a for f, a in tooltip_pairs_d if f in g_sp.columns]

folium.GeoJson(
    g_sp, name='Setores SP',
    style_function=style_d,
    tooltip=folium.features.GeoJsonTooltip(fields=fields_d, aliases=aliases_d,
                                            sticky=True, max_width=420),
).add_to(m_d)

if cmap_sp is not None:
    cmap_sp.add_to(m_d)

# Reusa a mesma legenda do Mapa C.
m_d.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl(collapsed=False).add_to(m_d)

mapa_d_path = MAPAS_DIR / f'mapa_sp_{MUN_SP}.html'
m_d.save(str(mapa_d_path))
print(f'Mapa D salvo: {mapa_d_path}  ({mapa_d_path.stat().st_size/1e6:.1f} MB)')
m_d


## Notas operacionais

- **Re-rodar este notebook** sempre que o parquet do nb 4 for atualizado (smoke → full Brasil).
- **Mudar recorte**: ajuste `UF_RECORTE` e `MUN_RECORTE` na célula `config-recorte` e re-execute as células a partir do mapa que quer atualizar.
- **Smoke vs full**:
  - Com smoke (RO+AC+RR): Mapa A mostra ~80 municípios; B/C funcionam pra qualquer município em RO/AC/RR.
  - Com full Brasil: Mapa A mostra ~5.570 municípios; B/C funcionam pra qualquer recorte do Brasil.
- **Tamanho dos HTMLs**: o Mapa A é o crítico — se ficar >50 MB, aumente `SIMPLIFY_TOLERANCE` (ex.: 0,01 ou 0,02). O Mapa B/C dependem do recorte escolhido.
- **Abrir HTMLs standalone**: duplo clique no arquivo em `saida_etl_final_brasil/mapas/`. Tudo embedded (Leaflet + dados), sem dependência externa.
- **Setores cinza**: legítimos sem renda (`origem_renda` começa com `sem_dado_legitimo_`) — ver [`RELATORIO_ANALISE_SP.md`](../RELATORIO_ANALISE_SP.md) Seção 4 para a discussão de por que 100% não é o teto saudável.